# Brain-to-Text: Skip-Diphone + Temporal Smoothness

**Project**: Context-Aware Neural Speech Decoding with Skip-Diphone Auxiliary Supervision and Temporal Smoothness Regularization  
**Author**: Jiayu (Jarrod) Yang — Introduction to Deep Learning, Spring 2026  
**Proposal**: [../docs/proposal.pdf](../docs/proposal.pdf)

---

## Methodology

- The pkl `train` bucket is partitioned 90/10 into train/dev (`dev_stride=10`,
  every 10th trial within each day).
- `best_dev.pt` is selected by dev PER. The pkl `test` split is evaluated
  every `eval_test_every` epochs only as a tracking signal.
- This notebook reports the **test PER at the best-dev epoch** as the
  headline acoustic metric (read from `<run>/summary.json`).
- Multi-seed runs (`scripts/multi_seed.sh`) are aggregated as mean ± std.

## Ablation Variants

| Variant | Components |
|---------|------------|
| A | Mono CTC only (NPTL baseline) |
| B | Diphone + marginalization (DCoND) |
| C | B + smoothness loss |
| D | B + skip-diphone auxiliary head |
| E | B + skip-diphone + smoothness (full model) |

## Training Objective (Eq. 1)

$$\mathcal{L}_{\text{total}} = \mathcal{L}^{\text{CTC}}_{\text{phoneme}} + \alpha\,\mathcal{L}^{\text{CTC}}_{\text{std-diphone}} + \beta\,\mathcal{L}^{\text{CTC}}_{\text{skip-diphone}} + \lambda\,\mathcal{L}_{\text{smooth}}$$

$$\mathcal{L}_{\text{smooth}} = \frac{1}{B}\sum_i \frac{1}{T_i-1}\sum_{t=2}^{T_i}\|p_t - p_{t-1}\|_2^2$$


In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EXP_ROOT = Path('../experiments')
# Run name: variant_X_alphaA_betaB_lamL[_seedS]   (seed group is optional for
# backward compatibility with runs trained before the multi-seed protocol).
RUN_RE = re.compile(
    r'variant_(?P<variant>[A-E])'
    r'_alpha(?P<alpha>[0-9.]+)'
    r'_beta(?P<beta>[0-9.]+)'
    r'_lam(?P<lam>[0-9.eE+-]+)'
    r'(?:_seed(?P<seed>[0-9]+))?'
)

def list_runs():
    if not EXP_ROOT.exists():
        return []
    runs = []
    for d in sorted(EXP_ROOT.iterdir()):
        if not d.is_dir():
            continue
        m = RUN_RE.fullmatch(d.name)
        if not m:
            continue
        loss_path = d / 'loss.json'
        if not loss_path.exists():
            continue
        log = json.loads(loss_path.read_text())
        if not log:
            continue
        # Headline metrics (legacy log entries used `val_per`; new ones use
        # `dev_per` / `test_per`). Fall back gracefully.
        def per_field(e, *names):
            for n in names:
                if n in e:
                    return e[n]
            return None

        dev_pers = [per_field(e, 'dev_per', 'val_per') for e in log]
        dev_pers = [p for p in dev_pers if p is not None]
        if not dev_pers:
            continue
        best_dev_per = min(dev_pers)
        best_dev_epoch_idx = dev_pers.index(best_dev_per)

        # Read summary.json if available; that's the source of truth for the
        # 'test PER at best-dev epoch' headline number.
        summary_path = d / 'summary.json'
        if summary_path.exists():
            summary = json.loads(summary_path.read_text())
            test_at_best = summary.get('test_per_at_best_dev')
        else:
            # Old runs: try test_per from the same epoch as best dev.
            entry = log[best_dev_epoch_idx]
            test_at_best = entry.get('test_per')

        runs.append({
            'run': d.name,
            'variant': m.group('variant'),
            'alpha': float(m.group('alpha')),
            'beta': float(m.group('beta')),
            'lambda': float(m.group('lam')),
            'seed': int(m.group('seed')) if m.group('seed') else None,
            'best_dev_per': best_dev_per,
            'test_per_at_best_dev': test_at_best,
            'log': log,
        })
    return runs

runs = list_runs()
print(f'Found {len(runs)} runs')
for r in runs:
    test_at = r['test_per_at_best_dev']
    test_str = f'{test_at*100:6.2f}%' if test_at is not None else '   --  '
    print(f"  {r['run']:<70s}  dev {r['best_dev_per']*100:6.2f}%  test@best {test_str}")

## 1. Ablation Table (A–E) with Multi-Seed Aggregation

Runs that share `(variant, alpha, beta, lambda)` are grouped over seeds and
reported as mean ± std. The headline number is **test PER at the best-dev
epoch**, not the minimum dev PER itself.

In [ ]:
DESCRIPTIONS = {
    'A': 'Mono CTC (NPTL baseline)',
    'B': 'Diphone + marginalization (DCoND)',
    'C': 'B + smoothness',
    'D': 'B + skip-diphone aux',
    'E': 'B + skip-diphone + smoothness (full)',
}

def group_by_config(runs):
    groups = {}
    for r in runs:
        key = (r['variant'], r['alpha'], r['beta'], r['lambda'])
        groups.setdefault(key, []).append(r)
    return groups

def fmt_mean_std(values, scale=100.0):
    arr = np.array([v for v in values if v is not None], dtype=float)
    if arr.size == 0:
        return float('nan'), float('nan'), 0
    return arr.mean() * scale, arr.std(ddof=0) * scale, arr.size

def best_group_per_variant(groups):
    """Pick the (alpha, beta, lambda) cell with the lowest mean test_per_at_best_dev (falling back to mean dev PER)."""
    by_var = {}
    for key, group in groups.items():
        v = key[0]
        test_pers = [g['test_per_at_best_dev'] for g in group]
        if all(t is None for t in test_pers):
            score = np.mean([g['best_dev_per'] for g in group])
        else:
            score = np.mean([t for t in test_pers if t is not None])
        if v not in by_var or score < by_var[v][0]:
            by_var[v] = (score, key, group)
    return {v: (key, group) for v, (_, key, group) in by_var.items()}

def read_wer(run_dir):
    p = EXP_ROOT / run_dir
    summary = p / 'wer_summary.json'
    if summary.exists():
        return json.loads(summary.read_text()).get('wer', np.nan)
    csv_path = p / 'wer_sweep.csv'
    if csv_path.exists():
        sweep = pd.read_csv(csv_path)
        if not sweep.empty:
            return float(sweep['WER'].min())
    return np.nan

groups = group_by_config(runs)
best = best_group_per_variant(groups)

rows = []
for v in 'ABCDE':
    if v not in best:
        rows.append({'Variant': v, 'Description': DESCRIPTIONS[v],
                     'N seeds': 0, 'dev PER %': np.nan, 'test@best PER %': np.nan,
                     'WER %': np.nan, 'lambda': np.nan, 'alpha': np.nan, 'beta': np.nan})
        continue
    (variant, alpha, beta, lam), group = best[v]
    dev_mean, dev_std, n = fmt_mean_std([g['best_dev_per'] for g in group])
    test_mean, test_std, _ = fmt_mean_std([g['test_per_at_best_dev'] for g in group])
    # WER: use the best run within the group (lowest dev) for the WER lookup.
    best_in_group = min(group, key=lambda g: g['best_dev_per'])
    wer = read_wer(best_in_group['run'])
    rows.append({
        'Variant': v,
        'Description': DESCRIPTIONS[v],
        'N seeds': n,
        'dev PER %': f'{dev_mean:.2f} ± {dev_std:.2f}' if n > 1 else f'{dev_mean:.2f}',
        'test@best PER %': (f'{test_mean:.2f} ± {test_std:.2f}' if n > 1 else
                            (f'{test_mean:.2f}' if not np.isnan(test_mean) else '—')),
        'WER %': f'{wer*100:.2f}' if not np.isnan(wer) else '—',
        'lambda': lam, 'alpha': alpha, 'beta': beta,
    })

df = pd.DataFrame(rows)
df

## 2. λ Sweep (Smoothness Weight)

Per-(λ) mean ± std across seeds. Plots both dev and test@best curves.

In [ ]:
def lambda_curve(runs, variant, metric='test_per_at_best_dev'):
    by_lam = {}
    for r in runs:
        if r['variant'] != variant:
            continue
        if metric == 'test_per_at_best_dev' and r[metric] is None:
            continue
        by_lam.setdefault(r['lambda'], []).append(r[metric])
    lams = sorted(by_lam)
    means = [float(np.mean(by_lam[l])) * 100 for l in lams]
    stds  = [float(np.std(by_lam[l])) * 100 for l in lams]
    return lams, means, stds

fig, ax = plt.subplots(figsize=(6, 4))
for variant, marker in [('C', 'o'), ('E', 's')]:
    lams, means, stds = lambda_curve(runs, variant, metric='test_per_at_best_dev')
    if not lams:
        # Fall back to dev PER if no test_per_at_best_dev available.
        lams, means, stds = lambda_curve(runs, variant, metric='best_dev_per')
    if not lams:
        continue
    ax.errorbar(lams, means, yerr=stds, marker=marker, capsize=3,
                label=f'Variant {variant}')
ax.set_xscale('symlog', linthresh=1e-4)
ax.set_xlabel('lambda (smoothness weight)')
ax.set_ylabel('PER (%)  (test @ best dev)')
ax.set_title('Effect of smoothness weight on PER (mean ± std across seeds)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig('../experiments/lambda_sweep.pdf', bbox_inches='tight')
plt.show()

## 3. β Sweep (Skip-Diphone Weight)

Variant D, β ∈ {0.05, 0.1, 0.2, 0.3} (`scripts/beta_sweep.sh`).

In [ ]:
d_runs = [r for r in runs if r['variant'] == 'D']
if d_runs:
    by_beta = {}
    for r in d_runs:
        m = r['test_per_at_best_dev'] if r['test_per_at_best_dev'] is not None else r['best_dev_per']
        by_beta.setdefault(r['beta'], []).append(m)
    betas = sorted(by_beta)
    means = [float(np.mean(by_beta[b])) * 100 for b in betas]
    stds  = [float(np.std(by_beta[b])) * 100 for b in betas]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(betas, means, yerr=stds, marker='D', color='C2', capsize=3)
    ax.set_xlabel('beta (skip-diphone CTC weight)')
    ax.set_ylabel('PER (%)')
    ax.set_title('Variant D: PER vs. beta')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../experiments/beta_sweep.pdf', bbox_inches='tight')
    plt.show()
    pd.DataFrame({'beta': betas, 'PER %': means, '± std': stds})
else:
    print('No Variant D runs found yet. Run scripts/beta_sweep.sh first.')

## 4. WER Sweep Heatmap (acoustic_scale × blank_penalty)

Reads `<best-E run>/wer_sweep.csv` written by `scripts/wer_sweep.py`.

In [ ]:
VARIANT_FOR_HEATMAP = 'E'
if VARIANT_FOR_HEATMAP in best:
    _, group = best[VARIANT_FOR_HEATMAP]
    best_in_group = min(group, key=lambda g: g['best_dev_per'])
    csv_path = EXP_ROOT / best_in_group['run'] / 'wer_sweep.csv'
    if csv_path.exists():
        sweep = pd.read_csv(csv_path)
        pivot = sweep.pivot(index='blank_penalty', columns='acoustic_scale', values='WER') * 100
        fig, ax = plt.subplots(figsize=(7, 4))
        im = ax.imshow(pivot.values, aspect='auto', origin='lower', cmap='viridis_r')
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f'{c:.2f}' for c in pivot.columns])
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels([f'{r:.2f}' for r in pivot.index])
        ax.set_xlabel('acoustic_scale')
        ax.set_ylabel('blank_penalty')
        ax.set_title(f'WER % heatmap (variant {VARIANT_FOR_HEATMAP}: {best_in_group["run"]})')
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                ax.text(j, i, f'{pivot.values[i, j]:.1f}',
                        ha='center', va='center', color='white', fontsize=9)
        plt.colorbar(im, ax=ax, label='WER %')
        plt.tight_layout()
        plt.savefig('../experiments/wer_heatmap.pdf', bbox_inches='tight')
        plt.show()
    else:
        print(f'No wer_sweep.csv at {csv_path}. Run scripts/wer_sweep.py first.')
else:
    print(f'No variant {VARIANT_FOR_HEATMAP} runs found yet.')

## 5. Training Curves

Train PER (lighter) vs. dev PER (heavier) per variant — useful for spotting
overfit (train PER drops while dev PER stalls).

In [ ]:
fig, (ax_l, ax_p) = plt.subplots(1, 2, figsize=(11, 4))
for v in 'ABCDE':
    if v not in best:
        continue
    _, group = best[v]
    r = min(group, key=lambda g: g['best_dev_per'])
    log = r['log']
    epochs = [e['epoch'] for e in log]
    dev_loss = [e.get('dev_loss', e.get('val_loss')) for e in log]
    train_per = [e.get('train_per', None) for e in log]
    dev_per = [e.get('dev_per', e.get('val_per')) for e in log]
    ax_l.plot(epochs, dev_loss, label=f'Variant {v}')
    if any(p is not None for p in train_per):
        ax_p.plot(epochs, [p*100 if p is not None else None for p in train_per],
                  alpha=0.4, linestyle='--', color=f'C{ord(v)-ord("A")}')
    ax_p.plot(epochs, [p*100 for p in dev_per], label=f'Variant {v}',
              color=f'C{ord(v)-ord("A")}')

ax_l.set_xlabel('Epoch'); ax_l.set_ylabel('Dev CTC Loss')
ax_l.grid(True, alpha=0.3); ax_l.legend()
ax_p.set_xlabel('Epoch'); ax_p.set_ylabel('PER (%)')
ax_p.grid(True, alpha=0.3); ax_p.legend()
ax_p.set_title('solid = dev PER,  dashed = train PER')
plt.tight_layout()
plt.savefig('../experiments/training_curves.pdf', bbox_inches='tight')
plt.show()